# 02 · ISI genérica

**Objetivo.** Mostrar y validar el bloque de ISI genérica (Parte A de la guía, pasos A1 a A5):

- el **FIR** complejo causal de hasta 32 taps, con energía normalizada (decisión 6);
- los **perfiles**: eco, gaussiano, Bessel y Butterworth (diseñados en frecuencia desde el prototipo analógico, nunca con la transformación bilineal) y respuesta medida;
- la **ISI que produce cada perfil**, medida como SNR equivalente (Tabla 4 del PDF), y el efecto del retardo fraccionario de los perfiles causales;
- su efecto sobre señales QAM16 y PAM4.

Todo sale de `emulador/isi/` (`fir.py`, `profiles.py`, `characterize.py`) y del bloque `emulador/blocks/isi.py`, el mismo código que usan los tests (`tests/test_isi.py`). Referencias: `referencia_guia_ISI_CD.md` Parte A; PDF *Fundamentos* §3 (Ec. 8 a 12, Figs. 1 y 2, Tablas 2 y 4).

In [ ]:
%load_ext autoreload
%autoreload 2

from functools import partial

import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

from emulador import plots
from emulador.blocks import ISI
from emulador.config import Scenario
from emulador.isi import K_rule, analog, analog_prototype, energy_captured, gaussian, isi_snr_eq
from emulador.metrics import group_delay, isi_power_ratio, peak_distortion, symbol_rate_response
from emulador.rx import matched_filter, recover
from emulador.stimuli import comm_signal

plots.use_style()

## 1. Teoría

**Modelo.** La ISI se emula con un FIR complejo causal a la frecuencia de muestreo:

$$
y[n] = \sum_{k=0}^{K-1} h[k]\,x[n-k], \qquad K \le 32, \qquad \sum_k |h[k]|^2 = 1 .
$$

La normalización por energía (Ec. 9) hace que la ISI **redistribuya** la potencia entre símbolos vecinos sin cambiarla: la SNR la fija el bloque AWGN. Todos los perfiles solo cambian los $h[k]$ que se cargan; la estructura del hardware es siempre la misma.

**Perfiles pasa-bajos.** Se especifican con el ancho de banda a −3 dB, $B = B_{rel}\cdot R_s$, y se diseñan por **muestreo en frecuencia**: se evalúa la respuesta deseada en la grilla de una FFT de $L \gg K$ puntos, se antitransforma y se trunca.

- Gaussiano: $H_G(f) = \exp\!\left(-\tfrac{\ln 2}{2}\,f^2/B^2\right)$, simétrico (se centra antes de truncar). Es el perfil de validación.
- Bessel y Butterworth: prototipo analógico de orden $m$, con un pre-retardo de $d$ muestras ($d = 3$ con $K = 16$, $d = 6$ con $K = 32$) para no perder la oscilación previa al pulso. El Bessel tiene fase casi lineal; el Butterworth agrega ISI por fase (perfil de estrés).
- Regla de K: 32 si $B \le 0{,}4\,R_s$ y $m \ge 4$; 16 si no. **Perfil principal**: Bessel $m = 4$, $B = 0{,}6\,R_s$, $K = 16$.

**Retardo** (decisión 8). El cursor es $k_0 = \arg\max|h[k]|$. `process_hw` es el FIR causal (sale retrasado $k_0$); `process` devuelve la salida alineada. El retardo **fraccionario** no se corrige: le toca a la recuperación de temporización del receptor.

**SNR equivalente.** Con $p[n]$ la respuesta TX (RRC) → canal → RX (RRC) a la tasa de símbolos y $n_0$ el cursor,

$$
\mathrm{SNR}_{eq} = -10\log_{10}\frac{\sum_{n\ne n_0}|p[n]|^2}{|p[n_0]|^2}.
$$

Se compara directamente con la SNR del enlace (mayor es mejor).

## 2. Parámetros

In [ ]:
esc = Scenario()          # 32 GBd, 2 sps -> fs = 64 GSa/s, roll-off 0,1
esc16 = Scenario(sps=16)  # solo para el ojo y el instante óptimo
Rs, fs = esc.Rs, esc.fs
B_RELS = (0.4, 0.5, 0.6)
KINDS = ("gaussian", "bessel", "butter")
LABEL = {"gaussian": "Gaussiano", "bessel": "Bessel m=4", "butter": "Butterworth m=4"}
N_SYM = 4096
SEED = 1

esc.to_dict()

## 3. Perfiles con B = 0,6·Rs (Fig. 2 del PDF)

Los tres perfiles con K = 16 a 64 GSa/s. El retardo de grupo se muestra en la banda $|f| \le B$, en muestras, descontando el retardo medio.

In [ ]:
NF = 8192
f = np.fft.fftshift(np.fft.fftfreq(NF, 1 / fs))
B6 = 0.6 * Rs
fig, axs = plt.subplots(1, 3, figsize=(16, 4.3))
summary_06 = {}
for i, kind in enumerate(KINDS):
    blk = ISI(profile=kind, B_rel=0.6, order=4)
    h = blk.coefficients()["h"].real
    H = np.fft.fftshift(np.fft.fft(h, NF))
    f_gd, tau = group_delay(np.fft.fft(h, NF), fs)
    band = np.abs(f_gd) <= B6
    tau_b = tau[band] * fs
    k = np.arange(h.size)
    H_at = lambda f_: abs(np.sum(h * np.exp(-2j * np.pi * f_ * k / fs)))  # noqa: E731
    summary_06[kind] = dict(K=h.size, cursor=blk.delay_samples, energy=np.sum(h**2),
                            db_B=20 * np.log10(H_at(B6) / H_at(0.0)), gd_var=np.ptp(tau_b))
    pos = f >= 0
    axs[0].plot(f[pos] / 1e9, 20 * np.log10(np.abs(H[pos]) / np.abs(H[NF // 2])), label=LABEL[kind])
    axs[1].plot(f_gd[band] / 1e9, tau_b - tau_b.mean(), label=LABEL[kind])
    axs[2].plot(k + 0.08 * (i - 1), h, "o-", ms=4, label=LABEL[kind])
axs[0].axvline(B6 / 1e9, ls=":", color="k")
axs[0].axhline(-3, ls=":", color="k")
axs[0].set_ylim(-30, 2)
axs[0].set_xlabel("f [GHz]")
axs[0].set_title(f"|H(f)| [dB] (B = 0,6 Rs = {B6 / 1e9:.1f} GHz)")
axs[1].set_xlabel("f [GHz]")
axs[1].set_title("Retardo de grupo en banda - media [muestras]")
axs[2].set_xlabel("k")
axs[2].set_title("Coeficientes h[k] (K = 16)")
for ax in axs:
    ax.legend()
plots.save_fig(fig, "02_perfiles_B06")

print("perfil            K  cursor  energía   |H(B)| [dB]  var. retardo [muestras]")
for kind, s in summary_06.items():
    print(f"{LABEL[kind]:16s} {s['K']:3d} {s['cursor']:6d}  {s['energy']:.6f}   {s['db_B']:9.2f}  {s['gd_var']:12.2f}")

**Qué muestra la figura.**
- **Magnitud**: los tres perfiles cruzan −3 dB en B = 19,2 GHz (−3,00 / −3,07 / −3,14 dB). Fuera de la banda el Butterworth cae mucho más rápido.
- **Retardo de grupo**: el gaussiano y el Bessel son planos (variación de 0,02 y 0,06 muestras): tienen fase lineal y su único efecto es atenuar. El Butterworth varía 0,77 muestras en la banda: es el único que agrega ISI por fase.
- **Coeficientes**: el gaussiano es simétrico y centrado (cursor en k = 8); el Bessel y el Butterworth son causales, con el cursor en k = 4 y k = 5, después del pre-retardo d = 3.

## 4. Orden del filtro (Tabla 2 y Fig. 1 del PDF)

Prototipos analógicos con B normalizado a 1 (el eje es f/B y t·B). La Tabla 2 da la variación del retardo de grupo en $|f| \le B$ en períodos de símbolo para B = 0,6·Rs, es decir, dividida por 0,6.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4.3))
f_n = np.linspace(1e-4, 3, 3000)
t_n = np.linspace(0, 3, 3001)
table2 = {}
for kind, ls in (("bessel", "-"), ("butter", "--")):
    for i, m in enumerate((2, 3, 4, 5, 6)):
        b, a = analog_prototype(kind, 1.0, m)
        _, H = signal.freqs(b, a, worN=2 * np.pi * f_n)
        tau = -np.gradient(np.unwrap(np.angle(H)), 2 * np.pi * f_n)
        inb = f_n <= 1
        _, H2 = signal.freqs(b, a, worN=[2 * np.pi * 2.0])
        _, y_long = signal.step((b, a), T=np.linspace(0, 40, 40001))
        table2[kind, m] = (20 * np.log10(abs(H2[0])), np.ptp(tau[inb]) / 0.6, 100 * (y_long.max() / y_long[-1] - 1))
        if m in (2, 4, 6):
            color = f"C{(2, 4, 6).index(m)}"
            name = f"{'Bessel' if kind == 'bessel' else 'Butterworth'} m={m}"
            axs[0].plot(f_n, 20 * np.log10(np.abs(H)), ls, color=color, label=name)
            axs[1].plot(f_n[inb], tau[inb] - tau[inb][0], ls, color=color, label=name)
            _, y = signal.step((b, a), T=t_n)
            axs[2].plot(t_n, y, ls, color=color, label=name)
axs[0].set_ylim(-60, 3)
axs[0].set_xlabel("f/B")
axs[0].set_title("|H(f)| [dB]")
axs[1].set_xlabel("f/B")
axs[1].set_title("Variación del retardo de grupo [1/B]")
axs[2].set_xlabel("t·B")
axs[2].set_title("Respuesta al escalón")
axs[0].legend(fontsize=9)
plots.save_fig(fig, "02_orden_filtro")

print(" m |  Bessel: aten 2B  var.ret [T]  sobrepico |  Butterworth: aten 2B  var.ret [T]  sobrepico")
for m in (2, 3, 4, 5, 6):
    be, bu = table2["bessel", m], table2["butter", m]
    print(f" {m} |  {be[0]:13.1f}  {be[1]:11.2f}  {be[2]:8.1f} % |  {bu[0]:18.1f}  {bu[1]:11.2f}  {bu[2]:8.1f} %")

**Qué muestra la figura.** El Butterworth gana unos 6 dB de atenuación en 2B por cada orden (de −12,3 dB con m = 2 a −36,1 dB con m = 6), a costa de más variación del retardo de grupo (0,08 a 0,70 T) y más sobrepico al escalón (4,3 a 14,3 %). El Bessel mantiene el retardo plano y un sobrepico menor que 1 % con cualquier orden, pero su atenuación cerca de la banda mejora poco (−9,8 a −14,2 dB). Reproduce la Tabla 2 del PDF; m = 4 es un valor representativo de un componente real.

## 5. Por qué el diseño en frecuencia

**Gaussiano: no muestrear $h(t)$.** La antitransformada continua es $h(t) = B\sqrt{2\pi/\ln 2}\,\exp(-2\pi^2B^2t^2/\ln 2)$. Muestrearla en $t = kT_s$ produce aliasing en frecuencia cuando $H_G$ no es despreciable en $f_s/2$, y el filtro queda más ancho de lo pedido.

**Bessel: no usar la transformación bilineal** (decisión 6). Su pre-distorsión hace caer el −3 dB en B, pero comprime el eje de frecuencias cerca de Nyquist: a 2 sps con B = 0,6·Rs el Bessel pierde la fase lineal.

In [ ]:
# =============================================================================
# Parte A · Gaussiano: muestrear h(t) contra diseñar en frecuencia
# =============================================================================
def rel_db(h, f_, fs_):
    """|H(f_)| / |H(0)| en dB, con H(f) = Σ h[k]·exp(-j 2π f k Ts) (respuesta del FIR)."""
    k = np.arange(len(h))
    H = lambda x: abs(np.sum(np.asarray(h) * np.exp(-2j * np.pi * x * k / fs_)))  # noqa: E731
    return 20 * np.log10(H(f_) / H(0.0))

alias = {}
for B_rel in (0.3, 0.5, 0.6, 0.8):
    B = B_rel * Rs
    # Opción descartada: muestrear la respuesta al impulso CONTINUA del gaussiano
    # en t = k·Ts (invarianza al impulso). Lo que H_G tiene por encima de fs/2 se
    # pliega sobre la banda (aliasing) y el filtro queda más ancho de lo pedido.
    t = (np.arange(64) - 32) / fs
    h_t = B * np.sqrt(2 * np.pi / np.log(2)) * np.exp(-2 * np.pi**2 * B**2 * t**2 / np.log(2))
    # Opción usada: gaussian() diseña en frecuencia (muestrea H_G en la grilla
    # de la FFT, antitransforma y trunca a 16 taps).
    alias[B_rel] = (rel_db(h_t, B, fs), rel_db(gaussian(B, fs, 16), B, fs))
print("B/Rs   h(t) muestreado   diseño en frecuencia   (|H(B)/H(0)| en dB)")
for B_rel, (a_, d_) in alias.items():
    print(f"{B_rel:4.1f}   {a_:15.2f}   {d_:20.2f}")

# =============================================================================
# Parte B · Bessel m = 4, B = 0,6 Rs: tres versiones del mismo filtro
#   HP    : prototipo ANALÓGICO (la referencia, el filtro "real")
#   Hbil  : digital por TRANSFORMACIÓN BILINEAL (IIR)
#   Hfreq : FIR por DISEÑO EN FRECUENCIA (el que usa el emulador)
# Todas se evalúan en la misma grilla de frecuencias para compararlas.
# =============================================================================
B, m = 0.6 * Rs, 4
f_full = np.fft.fftfreq(NF, 1 / fs)                 # grilla de comparación [Hz], orden de np.fft

# --- 1) Prototipo analógico ---------------------------------------------------
# analog_prototype usa signal.bessel(m, 2πB, analog=True, norm="mag"):
# coeficientes (b, a) de H_P(s), con s en rad/s.
b, a = analog_prototype("bessel", B, m)
# signal.freqs evalúa un filtro ANALÓGICO H_P(jω) en las ω pedidas (rad/s).
_, HP = signal.freqs(b, a, worN=2 * np.pi * f_full)

# --- 2) TRANSFORMACIÓN BILINEAL  <<< ACÁ SE APLICA >>> --------------------------
# signal.bessel SIN analog=True diseña un filtro DIGITAL: internamente arma el
# prototipo analógico y lo pasa a z con la transformación bilineal
#     s -> (2/Ts)·(1 - z^-1)/(1 + z^-1),
# con pre-distorsión (prewarping) para que el -3 dB caiga en B.
# El corte se da normalizado a Nyquist: Wn = B / (fs/2) (0,6 a 2 sps).
# Devuelve (bz, az): un IIR de orden m (numerador y denominador en z), no un FIR.
bz, az = signal.bessel(m, B / (fs / 2), norm="mag")
# signal.freqz evalúa un filtro DIGITAL H(e^{jΩ}); Ω = 2π f / fs en rad/muestra.
_, Hbil = signal.freqz(bz, az, worN=2 * np.pi * f_full / fs)

# --- 3) Diseño en frecuencia (el del emulador) --------------------------------
# analog() evalúa H_P en la grilla de la FFT, agrega el pre-retardo d = 3,
# antitransforma y se queda con 16 taps: un FIR. Hfreq es su respuesta.
h_main = analog("bessel", B, fs, m, 16)
Hfreq = np.fft.fft(h_main, NF)

# --- Comparación en la banda |f| <= B -----------------------------------------
# Retardo de grupo de cada versión (en segundos).
fg, tauP = group_delay(HP, fs)
_, tauB = group_delay(Hbil, fs)
_, tauF = group_delay(Hfreq, fs)
band = np.abs(fg) <= B
sh = lambda H: np.fft.fftshift(H)  # noqa: E731     # ordena de -fs/2 a fs/2
# Magnitudes en dB, relativas a f = 0 (el prototipo ya vale 1 en f = 0).
magP = 20 * np.log10(np.abs(sh(HP)))
magB = 20 * np.log10(np.abs(sh(Hbil)) / np.abs(sh(Hbil))[NF // 2])
magF = 20 * np.log10(np.abs(sh(Hfreq)) / np.abs(sh(Hfreq))[NF // 2])
# Error de retardo contra el prototipo, en muestras:
#  - bilineal: no tiene pre-retardo, pero su retardo medio difiere del analógico;
#    se compara la VARIACIÓN (se le resta la media): mide la pérdida de fase lineal.
#  - frecuencia: se descuenta el pre-retardo d = 3.
dB_bil = (tauB - tauP)[band] * fs
dB_fr = (tauF - tauP)[band] * fs - 3
err = {"bilineal": (np.max(np.abs(magB - magP)[band]), np.max(np.abs(dB_bil - dB_bil.mean()))),
       "frecuencia": (np.max(np.abs(magF - magP)[band]), np.max(np.abs(dB_fr)))}

# --- Sobrepico al escalón -----------------------------------------------------
# Diseño en frecuencia a 16 sps (K y d ×8, mismo tiempo) para ver la forma continua:
# el escalón es la suma acumulada de la respuesta al impulso.
fs16 = 16 * Rs
step_fr = np.cumsum(np.pad(analog("bessel", B, fs16, m, 128, d=24), (0, 400)))
# Bilineal a 2 sps: se filtra un escalón con el IIR (lfilter con bz, az = recursión).
s2 = signal.lfilter(bz, az, np.ones(400))
ov = {"bilineal (2 sps)": 100 * (s2.max() / s2[-1] - 1),
      "frecuencia (16 sps)": 100 * (step_fr.max() / step_fr[-1] - 1)}

# --- Figura 02_frecuencia_vs_bilineal -----------------------------------------
# mismo color para cada método en los tres paneles
COL = {"prototipo": "0.6", "frecuencia": "C0", "bilineal": "C1"}  # prototipo en gris, por detrás
fig, axs = plt.subplots(1, 3, figsize=(16, 4.3))
fb = fg[band] / 1e9
axs[0].plot(fb, magP[band], color=COL["prototipo"], lw=5, zorder=1, label="prototipo analógico")
axs[0].plot(fb, magF[band], color=COL["frecuencia"], label="diseño en frecuencia")
axs[0].plot(fb, magB[band], "--", color=COL["bilineal"], label="bilineal")
axs[0].set_title("|H| en |f| ≤ B [dB]")
axs[1].plot(fb, dB_fr, color=COL["frecuencia"], label="diseño en frecuencia")
axs[1].plot(fb, dB_bil - dB_bil.mean(), "--", color=COL["bilineal"], label="bilineal")
axs[1].set_title("Error de retardo contra el prototipo [muestras]")
t2 = np.arange(40) / 2
axs[2].plot(t2, s2[:40] / s2[-1], "o--", ms=3, color=COL["bilineal"], label=f"bilineal: sobrepico {ov['bilineal (2 sps)']:.1f} %")
t16 = np.arange(320) / 16
axs[2].plot(t16 - 3 / 2, step_fr[:320] / step_fr[-1], color=COL["frecuencia"], label=f"frecuencia: sobrepico {ov['frecuencia (16 sps)']:.1f} %")
axs[2].set_xlim(0, 12)
axs[2].set_xlabel("t [símbolos]")
axs[2].set_title("Respuesta al escalón")
for ax in axs[:2]:
    ax.set_xlabel("f [GHz]")
for ax in axs:
    ax.legend(fontsize=9)
plots.save_fig(fig, "02_frecuencia_vs_bilineal")

for name, (e_mag, e_dly) in err.items():
    print(f"{name:10s}: error en banda {e_mag:.2f} dB, {e_dly:.2f} muestras")
for name, o in ov.items():
    print(f"sobrepico {name}: {o:.1f} %")

**Qué muestra la figura.** Bessel m = 4 con B = 0,6·Rs a 2 sps:
- **Magnitud**: el diseño en frecuencia se superpone con el prototipo analógico (0,06 dB de error); la transformación bilineal se separa 0,53 dB cerca de B.
- **Retardo**: el diseño en frecuencia queda a ±0,03 muestras del prototipo; la bilineal dibuja una parábola de ~1 muestra, es decir, pierde la fase lineal que justifica usar un Bessel.
- **Escalón**: 0,8 % de sobrepico (el del filtro real) contra 15 % con la bilineal.

Junto con la tabla del gaussiano (muestrear $h(t)$ corre el −3 dB a −2,55 dB con B = 0,5·Rs y a −0,33 dB con B = 0,8·Rs), respalda la decisión 6: todos los perfiles se diseñan en frecuencia.

## 6. SNR equivalente (Tabla 4 del PDF, decisión 22)

**Instante óptimo**: el perfil se diseña con 16 sps (K y d ×8) y se elige la mejor de las 16 fases. **Grilla fija**: a 2 sps, muestreando en el cursor entero, sin corregir el retardo fraccionario.

In [ ]:
TABLE_4 = {("gaussian", 0.4): 16.2, ("bessel", 0.4): 15.7, ("butter", 0.4): 12.2,
           ("gaussian", 0.5): 19.9, ("bessel", 0.5): 19.8, ("butter", 0.5): 18.9,
           ("gaussian", 0.6): 23.0, ("bessel", 0.6): 23.3, ("butter", 0.6): 25.9}
snr = {key: isi_snr_eq(*key) for key in TABLE_4}

print("perfil            B/Rs   PDF [dB]  óptimo [dB]  dif.   grilla fija [dB]")
for (kind, B_rel), ref in TABLE_4.items():
    opt, grid = snr[kind, B_rel]
    print(f"{LABEL[kind]:16s} {B_rel:5.1f}   {ref:7.1f}   {opt:10.1f}  {opt - ref:+5.2f}   {grid:15.1f}")

fig, axs = plt.subplots(1, 2, figsize=(12, 4.3), sharey=True)
w = 0.25
for i, kind in enumerate(KINDS):
    xs = np.arange(len(B_RELS)) + (i - 1) * w
    axs[0].bar(xs, [snr[kind, b][0] for b in B_RELS], w, label=LABEL[kind])
    axs[0].plot(xs, [TABLE_4[kind, b] for b in B_RELS], "k_", ms=14, mew=2)
    axs[1].bar(xs, [snr[kind, b][1] for b in B_RELS], w, label=LABEL[kind])
for ax, title in zip(axs, ("Instante óptimo (— : Tabla 4 del PDF)", "Grilla fija de 2 sps")):
    ax.set_xticks(range(len(B_RELS)), [f"B = {b} Rs" for b in B_RELS])
    ax.set_title(title)
axs[0].set_ylabel("SNR equivalente [dB]")
axs[0].legend()
plots.save_fig(fig, "02_snr_equivalente")

**Qué muestra la figura.**
- **Izquierda, instante óptimo**: las barras reproducen la Tabla 4 del PDF (marcas negras) con 0,07 dB de diferencia como máximo. Domina B: de 0,4 a 0,6·Rs se ganan 7 a 14 dB; con el mismo B, gaussiano y Bessel son casi iguales y el Butterworth es el más severo en banda angosta.
- **Derecha, grilla fija de 2 sps**: el gaussiano no cambia (es simétrico), pero el Bessel cae a ~10 dB con B = 0,4 y 0,5·Rs y el Butterworth a 6,6 dB con B = 0,6·Rs (pierde 19 dB). La diferencia es el retardo fraccionario de los perfiles causales, que el bloque no corrige y le toca al receptor (decisión 8).

## 7. Señal QAM16 y ojo PAM4 con el perfil principal (Bessel m = 4)

Fila de arriba: el emulador a 2 sps, `comm_signal` → `ISI` → `rx.recover`. El receptor elige la mejor fase **entera** y una ganancia compleja, pero no interpola: el retardo fraccionario del Bessel queda sin corregir (grilla fija). Fila de abajo: la misma cadena a 16 sps, con el perfil diseñado a esa tasa (K y d ×8) y búsqueda entre 16 fases: es el **instante óptimo**, y muestra solo la ISI. El ojo PAM4 va a 16 sps, después del filtro adaptado.

In [ ]:
def bessel_16sps(B_rel):
    # Bessel m = 4 diseñado a 16 sps, cubriendo el mismo tiempo que a 2 sps
    K2 = K_rule(B_rel * Rs, Rs, 4)
    d2 = 6 if K2 >= 32 else 3
    return analog("bessel", B_rel * Rs, esc16.fs, 4, K2 * 8, d=d2 * 8)

def apply_aligned(x, h):
    k0 = int(np.argmax(np.abs(h)))
    return np.convolve(x, h)[k0 : k0 + x.size]

sig_q = comm_signal(esc, mod="qam16", n_symbols=N_SYM, seed=SEED)
sig_q16 = comm_signal(esc16, mod="qam16", n_symbols=N_SYM, seed=SEED)
fig, axs = plt.subplots(2, 4, figsize=(18, 9.2))
evm_grid, evm_opt16 = {}, {}
for row, (sig_, tag) in enumerate(((sig_q, "2 sps"), (sig_q16, "16 sps"))):
    r0 = recover(sig_.samples, sig_)
    plots.constellation(axs[row, 0], r0.symbols, ref=r0.ref, title=f"Sin ISI ({tag}) · EVM {100 * r0.evm:.2f} %")
    for ax, B_rel in zip(axs[row, 1:], B_RELS):
        if row == 0:
            y = ISI(profile="bessel", B_rel=B_rel)(sig_.samples)
        else:
            y = apply_aligned(sig_.samples, bessel_16sps(B_rel))
        r = recover(y, sig_)
        (evm_grid if row == 0 else evm_opt16)[B_rel] = r.evm
        plots.constellation(ax, r.symbols, ref=r.ref, title=f"B = {B_rel} Rs ({tag}) · EVM {100 * r.evm:.1f} %")
plots.save_fig(fig, "02_constelaciones_qam16")

print("B/Rs   EVM grilla fija (2 sps)   EVM instante óptimo (16 sps)   10^(-SNReq/20)")
for B_rel in B_RELS:
    print(f"{B_rel:4.1f}   {100 * evm_grid[B_rel]:21.1f} %   {100 * evm_opt16[B_rel]:26.1f} %"
          f"   {100 * 10 ** (-snr['bessel', B_rel][0] / 20):12.1f} %")

sig_p = comm_signal(esc16, mod="pam4", n_symbols=N_SYM, seed=SEED)
sps16 = esc16.sps
fig, axs = plt.subplots(1, 4, figsize=(18, 4.3), sharey=True)
skip = 2 * sig_p.span * sps16
for ax, B_rel in zip(axs, (None,) + B_RELS):
    x16 = sig_p.samples if B_rel is None else apply_aligned(sig_p.samples, bessel_16sps(B_rel))
    y16 = matched_filter(x16, esc16, sig_p.span)
    y16 = y16 / np.sqrt(np.mean(np.abs(y16[::sps16]) ** 2))
    title = "Sin ISI" if B_rel is None else f"Bessel m=4, B = {B_rel} Rs"
    plots.eye(ax, y16, sps16, n_traces=250, offset=skip + sps16 // 2, title=title)
    for lv in np.array([-3, -1, 1, 3]) / np.sqrt(5):
        ax.axhline(lv, color="C1", lw=0.8, ls=":")
plots.save_fig(fig, "02_ojo_pam4", dpi=110)

**Qué muestran las figuras.**
- **Constelaciones**: arriba, el emulador a 2 sps con un receptor que solo elige fases enteras: EVM 30,0 / 29,8 / 12,1 % con B = 0,4 / 0,5 / 0,6·Rs (con 0,4 y 0,5 casi igual, porque domina el retardo fraccionario sin corregir). Abajo, en el instante óptimo (16 sps): 16,1 / 10,2 / 6,8 %, que coincide con lo que predice la SNR equivalente (16,4 / 10,3 / 6,9 %). La métrica determinista y la simulación son consistentes.
- **Ojo PAM4**: con B = 0,4·Rs el ojo está casi cerrado; con 0,5 y 0,6·Rs se abre, pero la apertura queda corrida a ~0,55 T: es el retardo fraccionario del Bessel causal, visible a simple vista.

## 8. Magnitudes derivadas (PDF §6.3)

Para cada perfil y B, en el instante óptimo: distorsión de pico $D_p = \sum_{n\ne n_0}|p[n]|/|p[n_0]|$, potencia de ISI relativa al cursor y variación de $|H_{ISI}(f)|$ dentro de la banda de la señal, $|f| \le (1+\rho)R_s/2$ (con el perfil a 2 sps).

In [ ]:
def p_optimal(kind, B_rel, m=4, oversample=8):
    K2 = K_rule(B_rel * Rs, Rs, m)
    d2 = 6 if K2 >= 32 else 3
    fs_f = esc.sps * oversample * Rs
    if kind == "gaussian":
        h_f = gaussian(B_rel * Rs, fs_f, K2 * oversample, L=8192)
    else:
        h_f = analog(kind, B_rel * Rs, fs_f, m, K2 * oversample, d=d2 * oversample, L=8192)
    sps_f = esc.sps * oversample
    ps = [symbol_rate_response(h_f, sps_f, esc.rolloff, phase=ph) for ph in range(sps_f)]
    return min(ps, key=isi_power_ratio)

edge = (1 + esc.rolloff) * Rs / 2
print("perfil            B/Rs    Dp     ISI/cursor [dB]   var. |H| en banda [dB]")
derived = {}
for kind in KINDS:
    for B_rel in B_RELS:
        p = p_optimal(kind, B_rel)
        h = ISI(profile=kind, B_rel=B_rel).coefficients()["h"]
        Hf = np.fft.fftshift(np.fft.fft(h, NF))
        inb = np.abs(f) <= edge
        derived[kind, B_rel] = (peak_distortion(p), 10 * np.log10(isi_power_ratio(p)), np.ptp(20 * np.log10(np.abs(Hf[inb]))))
        dp, isi_db, var_h = derived[kind, B_rel]
        print(f"{LABEL[kind]:16s} {B_rel:5.1f}  {dp:5.2f}  {isi_db:15.1f}   {var_h:20.1f}")

## 9. Validación

Los mismos criterios que `tests/test_isi.py`.

In [ ]:
for kind, s in summary_06.items():
    assert s["K"] == 16 and abs(s["energy"] - 1) < 1e-12
    assert abs(s["db_B"] + 3) < 0.3
assert summary_06["bessel"]["cursor"] == 4 and summary_06["gaussian"]["cursor"] == 8
# Tabla 2 (B = 0,6 Rs): atenuación en 2B ±0,1 dB y sobrepico ±0,1 %
for (kind, m), (att, ovs) in {("bessel", 4): (-13.4, 0.8), ("butter", 4): (-24.1, 10.8),
                              ("butter", 6): (-36.1, 14.3), ("bessel", 2): (-9.8, 0.4)}.items():
    assert abs(table2[kind, m][0] - att) < 0.1 and abs(table2[kind, m][2] - ovs) < 0.1
assert abs(alias[0.5][0] + 2.55) < 0.01 and abs(alias[0.8][0] + 0.33) < 0.01
assert all(abs(d_ + 3) < 0.3 for _, d_ in alias.values())
assert err["frecuencia"][0] < 0.1 and err["frecuencia"][1] < 0.05
assert err["bilineal"][0] > 0.5 and ov["bilineal (2 sps)"] > 10 > 1 > ov["frecuencia (16 sps)"]
for key, ref in TABLE_4.items():
    assert abs(snr[key][0] - ref) <= 0.2
assert snr["butter", 0.4][0] < snr["bessel", 0.4][0] - 3
assert evm_opt16[0.4] > evm_opt16[0.5] > evm_opt16[0.6]
assert all(evm_grid[b] > evm_opt16[b] for b in B_RELS)
for b in B_RELS:  # la EVM simulada en el instante óptimo coincide con la SNR equivalente
    assert abs(evm_opt16[b] / 10 ** (-snr['bessel', b][0] / 20) - 1) < 0.05
print("OK")

## 10. Conclusiones

- **El FIR** de hasta 32 taps con energía normalizada es la estructura única del hardware: todos los perfiles solo cambian los $h[k]$. La salida alineada y la cruda (`process_hw`) difieren en el cursor $k_0$.
- **Diseño en frecuencia**: los tres perfiles pasa-bajos cruzan −3 dB en B (−3,00 / −3,07 / −3,14 dB a 0,6·Rs). El Bessel diseñado así se aparta del prototipo analógico 0,06 dB y 0,03 muestras en la banda; con la transformación bilineal, 0,53 dB, ~1 muestra de retardo y 15 % de sobrepico al escalón (contra 0,8 % del filtro real). Muestrear la $h(t)$ del gaussiano corre el −3 dB (−2,55 dB a 0,5·Rs, −0,33 dB a 0,8·Rs). Esto respalda la decisión 6.
- **Orden m**: el Butterworth gana ~6 dB de atenuación en 2B por orden, a costa de variación del retardo (0,34 T con m = 4) y sobrepico (10,8 %); el Bessel mantiene la fase lineal con cualquier orden. Reproduce la Tabla 2.
- **ISI por perfil (Tabla 4, reproducida con ≤ 0,07 dB)**: el parámetro dominante es B (de 0,4 a 0,6·Rs se ganan 7 a 14 dB de SNR equivalente); con el mismo B, gaussiano y Bessel son casi iguales y el Butterworth es el más severo en banda angosta (12,2 dB a 0,4·Rs).
- **Retardo fraccionario**: los perfiles causales corren el instante óptimo (en el ojo PAM4 la apertura queda en ~0,55 T). Sin corregirlo (grilla fija de 2 sps), el Bessel pierde 5 a 10 dB y el Butterworth hasta 19 dB, y la EVM de QAM16 casi se duplica (Bessel 0,5·Rs: 29,8 % contra 10,2 %). Corregirlo le toca a la recuperación de temporización del receptor (decisión 8). En el instante óptimo, la EVM simulada coincide con la SNR equivalente determinista (16,1 / 10,2 / 6,8 % contra 16,4 / 10,3 / 6,9 %).

Figuras en `docs/golden_model/figuras/02_*.png`.